## Scaling & a realistic stack

**Scaling** looks trivial — run N copies of a service:

```bash
docker compose up -d --scale web=3
```

That starts `web-1`, `web-2`, `web-3`, all on the project network, all answering to the DNS name `web` (the embedded resolver round-robins between them). Great for **local load testing** and **stateless** tiers. But mind two hard limits: a service with `ports: ["8080:80"]` can only bind 8080 **once**, so only one replica starts — to scale a published service, drop the host-port pin and put a load balancer (Nginx, Traefik) in front. And **never `--scale` a stateful service** like Postgres — you'd get N containers fighting over one data dir. Scaling *across machines* is Swarm/Kubernetes, not single-host Compose.

**A realistic stack** ties the whole module together — the Flask app from module 02, now with Postgres storage and Redis cache:

```yaml
name: rt-demo
services:
  db:
    image: postgres:16
    environment:
      POSTGRES_PASSWORD: ${DB_PASSWORD:?DB_PASSWORD must be set}
    volumes: [pgdata:/var/lib/postgresql/data]
    healthcheck:
      test: ["CMD-SHELL", "pg_isready -U postgres"]
      interval: 5s
      retries: 5
  cache:
    image: redis:7-alpine
    healthcheck: { test: ["CMD", "redis-cli", "PING"], interval: 5s, retries: 5 }
  web:
    build: ./web
    ports: ["127.0.0.1:8080:8080"]      # localhost only — safe by default
    environment:
      DATABASE_URL: postgres://app:${DB_PASSWORD}@db:5432/app
      REDIS_URL: redis://cache:6379/0
    depends_on:
      db:    { condition: service_healthy }
      cache: { condition: service_healthy }
    restart: unless-stopped
volumes:
  pgdata: {}
```

Every idea from the module is here: `build` for our code and `image` for dependencies, name-based DNS (`db`, `cache`), a persistent volume, **`service_healthy`** startup order, a **`${DB_PASSWORD:?...}`** required variable, a **localhost-only** publish, and `restart: unless-stopped`. One file, `docker compose up`, and the whole app is running — which is exactly what Compose is for.